# Gate features

One row per slate (not per candidate). They feed the gate `w = sigmoid(a·log1p(history_length) + b·log1p(time_since_last_activity) + c·log1p(purchase_count) + bias)` that mixes Model 1 (short-term) and Model 2 (long-term): `score = w·P1 + (1-w)·P2`. For very short histories the gate is overridden to `w = 1` (Model 1 only).

**Inputs:** `data/train_sampled.parquet`, `val.parquet`, `test.parquet` (only the history and candidate columns are read).
**Outputs:** `features/gate_train.parquet`, `gate_val.parquet`, `gate_test.parquet`.

**History filter:** none (`FILTER_MODE = "none"`), the same setting as the Model 1 feature notebook. The filter code is kept so a sensitivity check can be rerun; keep the setting identical in both notebooks.

## 0. Setup

In [1]:
import os
import numpy as np, pandas as pd, pyarrow.parquet as pq

DATA = "data"
OUT = "features"
FILTER_MODE = "none"       # must match the Model 1 feature notebook ("none" | "in_slate" | "all")
MIN_AGE_DAYS = 0.05        # only used when FILTER_MODE != "none"
SHORT_HISTORY_PCTL = 10    # hard override w=1 below this train percentile of history length
os.makedirs(OUT, exist_ok=True)

# candidate_wid_list is only needed for the in_slate filter
HIST = ["slate_id", "candidate_wid_list", "history_wid_list", "history_type_list", "history_days_before"]


def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)


def slate_keys(df):
    """One int64 key per (slate, candidate): slate_id * 2**32 + wid."""
    lens = df["candidate_wid_list"].map(len).to_numpy()
    sid = np.repeat(df["slate_id"].to_numpy().astype(np.int64), lens)
    wid = np.concatenate(df["candidate_wid_list"].to_numpy()).astype(np.int64)
    return sid * (1 << 32) + wid


def clean_history(h, keys, mode=None, min_age=None):
    mode = FILTER_MODE if mode is None else mode
    min_age = MIN_AGE_DAYS if min_age is None else min_age
    if mode == "none":
        return h
    recent = h["days"].to_numpy() < min_age
    if mode == "all":
        return h[~recent]
    if mode == "in_slate":
        hk = h["slate_id"].to_numpy().astype(np.int64) * (1 << 32) + h["wid"].to_numpy()
        return h[~(recent & np.isin(hk, keys))]
    raise ValueError("mode must be 'in_slate', 'all' or 'none'")

## 1. Gate feature definitions
- `gate_history_length`: number of history events (all types) left after the history filter (with `FILTER_MODE = "none"`: all events).
- `gate_time_since_last_activity`: days since the most recent such event. Empty history -> filled with the maximum observed age (365) and flagged.
- `gate_purchase_count`: number of ORD events in history. Orders are observed over the full year, so unlike clicks and carts this count is comparable across users.
- `log1p` versions are what enter the sigmoid (long right tails).

In [2]:
def build_gate_features(df, keys=None, mode=None, min_age=None):
    n_rows = len(df)
    lens = df["history_wid_list"].map(len).to_numpy()
    h = pd.DataFrame({
        "slate_id": np.repeat(df["slate_id"].to_numpy(), lens),
        "row": np.repeat(np.arange(n_rows), lens),
        "wid": np.concatenate(df["history_wid_list"].to_numpy()).astype(np.int64),
        "typ": np.concatenate(df["history_type_list"].to_numpy()).astype(str),
        "days": np.concatenate(df["history_days_before"].to_numpy()).astype(np.float32),   # float32, same as Model 1
    })
    mode_ = FILTER_MODE if mode is None else mode
    if mode_ != "none" and keys is None:
        keys = slate_keys(df)
    h = clean_history(h, keys, mode, min_age)
    row, typ, days = h["row"].to_numpy(), h["typ"].to_numpy(), h["days"].to_numpy().astype(np.float64)

    length = np.bincount(row, minlength=n_rows)
    purchases = np.bincount(row[typ == "ORD"], minlength=n_rows)
    last = pd.Series(days).groupby(row).min().reindex(range(n_rows))       # NaN when no events left
    empty = length == 0
    last = last.fillna(365.0).to_numpy()

    out = pd.DataFrame({
        "slate_id": df["slate_id"].to_numpy(),
        "gate_history_length": length.astype(np.int32),
        "gate_time_since_last_activity": last.astype(np.float32),
        "gate_purchase_count": purchases.astype(np.int32),
        "gate_empty_history": empty.astype(np.int8),
    })
    out["gate_log_history_length"] = np.log1p(out["gate_history_length"]).astype(np.float32)
    out["gate_log_time_since_last"] = np.log1p(out["gate_time_since_last_activity"]).astype(np.float32)
    out["gate_log_purchase_count"] = np.log1p(out["gate_purchase_count"]).astype(np.float32)
    return out

## 2. Build for all splits

In [3]:
gate = {}
for name, fn in [("train", "train_sampled"), ("val1", "val1"), ("val2", "val2"),("test", "test")]:
    d = load(f"{DATA}/{fn}.parquet", columns=HIST)
    gate[name] = build_gate_features(d)
    print(name, gate[name].shape)
    del d

train (122083, 8)
val1 (17799, 8)
val2 (16924, 8)
test (17025, 8)


## 3. Short-history override threshold
The threshold is the 10th percentile of `gate_history_length` on **train only**. Recompute it whenever the history filter changes.

In [4]:
thr = int(np.percentile(gate["train"]["gate_history_length"], SHORT_HISTORY_PCTL))
print("short-history threshold (train 10th percentile):", thr)
for name, g in gate.items():
    g["gate_short_history"] = (g["gate_history_length"] <= thr).astype(np.int8)
    g.to_parquet(f"{OUT}/gate_{name}.parquet", index=False)
    print(name, "override share:", round(g["gate_short_history"].mean() * 100, 2), "% | empty histories:", int(g["gate_empty_history"].sum()))

short-history threshold (train 10th percentile): 10
train override share: 10.3 % | empty histories: 0
val1 override share: 11.17 % | empty histories: 0
val2 override share: 10.23 % | empty histories: 0
test override share: 10.23 % | empty histories: 0


## 4. Descriptive statistics

In [5]:
cols = ["gate_history_length", "gate_time_since_last_activity", "gate_purchase_count"]
for name, g in gate.items():
    print("=====", name)
    print(g[cols].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3).to_string())

===== train
       gate_history_length  gate_time_since_last_activity  gate_purchase_count
count           122083.000                     122083.000           122083.000
mean               153.341                          5.431               39.144
std                224.121                         22.007               69.764
min                  1.000                          0.000                0.000
10%                 10.000                          0.000                1.000
25%                 29.000                          0.001                4.000
50%                 78.000                          0.222               15.000
75%                188.000                          2.139               44.000
90%                375.000                          9.658              103.000
99%               1059.180                         89.301              326.000
max               4802.000                        363.959             2899.000
===== val1
       gate_history_length  g

## 5. Sanity checks

In [8]:
for name, g in gate.items():
    assert g["slate_id"].is_unique
    assert (g["gate_history_length"] >= 0).all() and (g["gate_purchase_count"] <= g["gate_history_length"]).all()
    assert g["gate_time_since_last_activity"].between(0, 366).all()
    assert not g.isna().any().any()
print("basic checks OK")

# brute-force check on a small sample (val), written independently of clean_history
d = load(f"{DATA}/val2.parquet", columns=HIST).head(200)
g = gate["val2"].set_index("slate_id")
thr_age = np.float32(MIN_AGE_DAYS)
def keep(wid, day, cands):
    if FILTER_MODE == "none":
        return True
    recent = np.float32(day) < thr_age
    if FILTER_MODE == "all":
        return not recent
    return not (recent and int(wid) in cands)
for sid, cands, wids, types, days in zip(d["slate_id"], d["candidate_wid_list"], d["history_wid_list"],
                                         d["history_type_list"], d["history_days_before"]):
    cs = set(int(c) for c in cands)
    k = [(t, np.float32(x)) for w, t, x in zip(wids, types, days) if keep(w, x, cs)]
    assert g.loc[sid, "gate_history_length"] == len(k)
    assert g.loc[sid, "gate_purchase_count"] == sum(t == "ORD" for t, _ in k)
    if k:
        assert abs(g.loc[sid, "gate_time_since_last_activity"] - min(x for _, x in k)) < 1e-4
print("brute-force check on 200 val slates OK")

# train vs val vs test distribution (they should look alike)
print(pd.DataFrame({n: g[["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]].mean()
                    for n, g in gate.items()}).round(3))

basic checks OK
brute-force check on 200 val slates OK
                          train   val1   val2   test
gate_log_history_length   4.266  4.230  4.293  4.265
gate_log_time_since_last  0.761  0.772  0.753  0.751
gate_log_purchase_count   2.695  2.658  2.717  2.697


### 5.1 Is `time since last activity` still informative?
With no history filter, most searches have an event just before the query. If nearly all values sit at almost zero, `log1p` of them is almost constant and the gate cannot use this feature.

In [ ]:
t = gate["train"]["gate_time_since_last_activity"]
print("share of slates with last activity < 1 min / < 15 min / < 1 h / < 1 d:",
      [round(float((t < x).mean()), 3) for x in (1/1440, 15/1440, 1/24, 1)])
print(t.describe(percentiles=[.1, .25, .5, .75, .9]).round(4).to_string())
print("std of log time since last (train):", round(float(gate["train"]["gate_log_time_since_last"].std()), 3))

share of slates with last activity < 1 min / < 15 min / < 1 h / < 1 d: [0.229, 0.396, 0.432, 0.641]
count    139882.0000
mean          5.4442
std          22.0358
min           0.0000
10%           0.0002
25%           0.0008
50%           0.2263
75%           2.1421
90%           9.6703
max         363.9587
std of log time since last (train): 1.099


### 5.2 Collinearity between the log gate features
The logistic gate is sensitive to collinear inputs. VIF below about 5 is fine.

In [ ]:
X = gate["train"][["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]]
corr = X.corr()
print(corr.round(2).to_string())
vif = pd.Series(np.diag(np.linalg.inv(corr.to_numpy())), index=corr.index)
print("\nVIF:", vif.round(2).to_dict())

                          gate_log_history_length  gate_log_time_since_last  gate_log_purchase_count
gate_log_history_length                      1.00                     -0.46                     0.75
gate_log_time_since_last                    -0.46                      1.00                    -0.19
gate_log_purchase_count                      0.75                     -0.19                     1.00

VIF: {'gate_log_history_length': 3.0, 'gate_log_time_since_last': 1.36, 'gate_log_purchase_count': 2.45}


## 6. Next steps
1. When Model 1 and Model 2 both produce scores, standardize the three log features with the train mean and standard deviation and fit `a, b, c, bias` on **val** (full slates) to maximise ranking quality of `w·P1 + (1-w)·P2`, then report on test.
2. Before that, a useful evidence table: Model 1's NDCG@10 by gate bucket (history length, purchase count, time since last activity). If Model 1 is relatively better for short histories, the gate has a reason to exist.